# MOD-02 · NB-03 — Visualising Clinical Distributions
### Health Analytics with Python · Module 02: EDA in Healthcare
---
**Learning objectives**
- Build histograms and KDE plots for lab values and clinical outcomes
- Annotate plots with clinical reference ranges and thresholds
- Compare distributions across groups with box plots and violin plots
- Combine multiple plot types into publication-ready multi-panel figures
- Export figures at print resolution (300 dpi)

**Estimated time:** 2 hours | **Level:** Beginner → Intermediate | **Libraries:** `pandas`, `numpy`, `matplotlib`, `seaborn`


## 1. Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from scipy.stats import skew
import warnings
warnings.filterwarnings('ignore')

# ── Plot defaults ─────────────────────────────────────────────────────────────
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.05)
plt.rcParams.update({
    'figure.dpi'     : 120,
    'savefig.dpi'    : 300,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'figure.facecolor': 'white',
})

# ── Dataset ───────────────────────────────────────────────────────────────────
np.random.seed(42); N=800
ages      = np.random.normal(62,16,N).clip(18,95).astype(int)
sexes     = np.random.choice(['M','F'],N,p=[0.48,0.52])
payers    = np.random.choice(['Medicare','Medicaid','Private','Self-pay','Other'],N,p=[0.40,0.22,0.28,0.07,0.03])
drg_codes = np.random.choice([470,291,392,690,871,193,247,603],N)
drg_labels= {470:'Major joint replacement',291:'Heart failure & shock',392:'Esophagitis/misc GI',
             690:'Kidney/UTI',871:'Septicemia',193:'Simple pneumonia',247:'Perc cardiovasc w stent',603:'Cellulitis'}
los_days  = np.random.gamma(2.5,1.8,N).clip(1,30).astype(int)
charges   = (los_days*np.random.uniform(1800,4200,N)).round(2)
charges[np.random.rand(N)<0.08]=np.nan
primary_dx= np.random.choice(['E11.9','I10','N18.3','I50.9','J18.9','A41.9','M16.11','N39.0'],N)
admit_type= np.random.choice(['Emergency','Elective','Urgent','Newborn'],N,p=[0.52,0.30,0.16,0.02])
np.random.seed(99)
df = pd.DataFrame({
    'patient_id':[f'PT-{str(i).zfill(5)}' for i in range(1,N+1)],
    'age':ages,'sex':sexes,'payer':payers,
    'drg_code':drg_codes,'drg_label':[drg_labels[d] for d in drg_codes],
    'primary_dx':primary_dx,'admit_type':admit_type,
    'los_days':los_days,'total_charge':charges,
    'diabetes':np.random.binomial(1,0.32,N),'hypertension':np.random.binomial(1,0.45,N),
    'ckd':np.random.binomial(1,0.15,N),'heart_failure':np.random.binomial(1,0.18,N),
    'readmitted_30':np.random.binomial(1,0.14,N),
})
df['age_group']=pd.cut(df['age'],[17,44,64,74,95],labels=['18-44','45-64','65-74','75+'])

# Synthetic lab values
np.random.seed(21)
df['glucose']    = np.random.normal(105,28,N).clip(50,400).round(1)
df['creatinine'] = np.random.gamma(1.6,0.6,N).clip(0.4,10).round(2)
df['hba1c']      = np.random.normal(6.8,1.4,N).clip(4,14).round(1)
df['wbc']        = np.random.gamma(4,2,N).clip(1,30).round(1)
df['hemoglobin'] = np.random.normal(12.8,2.1,N).clip(5,18).round(1)

print(f"Dataset ready: {df.shape}")
df[['glucose','creatinine','hba1c','wbc','hemoglobin']].describe().round(2)


## 2. Histograms with Clinical Reference Ranges

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle('Lab value distributions with clinical reference ranges', fontsize=13, y=1.02)

# ── Glucose ───────────────────────────────────────────────────────────────────
ax = axes[0]
sns.histplot(df['glucose'], bins=35, kde=True, color='#4878CF', ax=ax)
ax.axvspan(70, 100, alpha=0.12, color='green', label='Normal (70–100 mg/dL)')
ax.axvline(126, color='red', ls='--', lw=1.4, label='Diabetes threshold (126)')
ax.axvline(df['glucose'].median(), color='navy', ls=':', lw=1.4,
           label=f'Median ({df["glucose"].median():.0f})')
ax.set_xlabel('Glucose (mg/dL)'); ax.set_ylabel('Count')
ax.set_title('Glucose'); ax.legend(fontsize=8)

# ── Creatinine ────────────────────────────────────────────────────────────────
ax = axes[1]
sns.histplot(df['creatinine'], bins=35, kde=True, color='#D65F5F', ax=ax)
ax.axvspan(0.6, 1.2, alpha=0.12, color='green', label='Normal (0.6–1.2 mg/dL)')
ax.axvline(1.5, color='darkorange', ls='--', lw=1.4, label='Mild AKI threshold')
ax.axvline(df['creatinine'].median(), color='darkred', ls=':', lw=1.4,
           label=f'Median ({df["creatinine"].median():.2f})')
ax.set_xlabel('Creatinine (mg/dL)'); ax.set_ylabel('')
ax.set_title(f'Creatinine  (skew={skew(df["creatinine"]):.2f})')
ax.legend(fontsize=8)

# ── HbA1c ─────────────────────────────────────────────────────────────────────
ax = axes[2]
sns.histplot(df['hba1c'], bins=30, kde=True, color='#6ACC65', ax=ax)
ax.axvspan(4, 5.6,  alpha=0.12, color='green',     label='Normal (<5.7%)')
ax.axvspan(5.7, 6.4, alpha=0.12, color='orange',   label='Pre-diabetes (5.7–6.4%)')
ax.axvspan(6.4, 14,  alpha=0.08, color='red',      label='Diabetes (≥6.5%)')
ax.axvline(df['hba1c'].median(), color='darkgreen', ls=':', lw=1.4,
           label=f'Median ({df["hba1c"].median():.1f}%)')
ax.set_xlabel('HbA1c (%)'); ax.set_ylabel('')
ax.set_title('HbA1c'); ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig('/tmp/mod02/lab_histograms.png', bbox_inches='tight')
plt.show()
print("Saved: lab_histograms.png")


## 3. Box Plots — LOS by Diagnosis Group

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# ── LOS by DRG ───────────────────────────────────────────────────────────────
order = (df.groupby('drg_label')['los_days'].median()
           .sort_values(ascending=False).index.tolist())

sns.boxplot(
    data=df, y='drg_label', x='los_days',
    order=order, palette='Blues_r',
    flierprops={'marker':'o','markersize':3,'alpha':0.4},
    ax=axes[0]
)
axes[0].axvline(df['los_days'].median(), color='red', ls='--', lw=1.2,
                label=f'Overall median ({df["los_days"].median():.0f} days)')
axes[0].set_xlabel('Length of stay (days)')
axes[0].set_ylabel('')
axes[0].set_title('LOS distribution by DRG')
axes[0].legend(fontsize=9)

# ── Charge by payer ───────────────────────────────────────────────────────────
payer_order = (df.groupby('payer')['total_charge'].median()
                 .sort_values(ascending=False).index.tolist())
sns.boxplot(
    data=df.dropna(subset=['total_charge']),
    x='payer', y='total_charge', order=payer_order,
    palette='Oranges_r',
    flierprops={'marker':'o','markersize':3,'alpha':0.4},
    ax=axes[1]
)
axes[1].yaxis.set_major_formatter(
    plt.FuncFormatter(lambda x,_: f'${x/1000:.0f}k'))
axes[1].set_xlabel('Payer')
axes[1].set_ylabel('Total charge')
axes[1].set_title('Charge distribution by payer')

plt.tight_layout()
plt.savefig('/tmp/mod02/boxplots_los_charge.png', bbox_inches='tight')
plt.show()


## 4. Violin Plots — Age by Payer

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ── Age by payer ──────────────────────────────────────────────────────────────
payer_age_order = (df.groupby('payer')['age'].median()
                     .sort_values(ascending=False).index.tolist())
sns.violinplot(data=df, x='payer', y='age', order=payer_age_order,
               palette='Set2', inner='quartile', ax=axes[0])
axes[0].axhline(65, color='red', ls='--', lw=1.2, label='Medicare eligibility (65)')
axes[0].set_xlabel('Payer'); axes[0].set_ylabel('Age (years)')
axes[0].set_title('Age distribution by payer')
axes[0].legend(fontsize=9)

# ── LOS by admit type + sex (split violin) ────────────────────────────────────
admit_order = ['Emergency','Urgent','Elective','Newborn']
sns.violinplot(data=df, x='admit_type', y='los_days', hue='sex',
               order=admit_order, split=True, palette={'M':'#5B9BD5','F':'#ED7D31'},
               inner='quartile', ax=axes[1])
axes[1].set_xlabel('Admission type'); axes[1].set_ylabel('LOS (days)')
axes[1].set_title('LOS by admission type and sex (split)')
axes[1].legend(title='Sex', fontsize=9)

plt.tight_layout()
plt.savefig('/tmp/mod02/violin_plots.png', bbox_inches='tight')
plt.show()


## 5. Strip + Box Overlay (Beeswarm-style)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

sns.boxplot(data=df, x='age_group', y='los_days',
            palette='pastel', width=0.4, fliersize=0, ax=ax)
sns.stripplot(data=df, x='age_group', y='los_days',
              color='steelblue', alpha=0.25, jitter=True, size=3, ax=ax)

# Annotate group medians
for i, grp in enumerate(['18-44','45-64','65-74','75+']):
    med = df.loc[df['age_group']==grp,'los_days'].median()
    ax.text(i, med+0.3, f'{med:.0f}d', ha='center', va='bottom',
            fontsize=9, color='navy', fontweight='bold')

ax.set_xlabel('Age group'); ax.set_ylabel('Length of stay (days)')
ax.set_title('LOS by age group — box + individual observations')
plt.tight_layout()
plt.savefig('/tmp/mod02/stripbox_los_age.png', bbox_inches='tight')
plt.show()


## 6. Publication-Ready Multi-Panel Figure

In [ ]:
fig = plt.figure(figsize=(16, 10))
gs  = fig.add_gridspec(2, 3, hspace=0.40, wspace=0.35)

# Panel A — Glucose histogram
ax_a = fig.add_subplot(gs[0, 0])
sns.histplot(df['glucose'], bins=30, kde=True, color='#4878CF', ax=ax_a)
ax_a.axvspan(70,100,alpha=0.12,color='green')
ax_a.axvline(126,color='red',ls='--',lw=1.2)
ax_a.set_title('A  Glucose distribution', fontweight='bold')
ax_a.set_xlabel('Glucose (mg/dL)')

# Panel B — LOS by payer (box)
ax_b = fig.add_subplot(gs[0, 1])
po = df.groupby('payer')['los_days'].median().sort_values(ascending=False).index
sns.boxplot(data=df,x='payer',y='los_days',order=po,palette='Blues_r',
            fliersize=2,ax=ax_b)
ax_b.set_title('B  LOS by payer', fontweight='bold')
ax_b.set_xlabel(''); ax_b.set_ylabel('LOS (days)')
ax_b.tick_params(axis='x',rotation=20)

# Panel C — Readmission rate bar
ax_c = fig.add_subplot(gs[0, 2])
rates = df.groupby('payer')['readmitted_30'].mean()*100
rates = rates.sort_values(ascending=False)
colors = ['#D65F5F' if r>15 else '#4878CF' for r in rates]
bars = ax_c.bar(rates.index, rates.values, color=colors, edgecolor='white', linewidth=0.5)
ax_c.axhline(rates.mean(),color='black',ls=':',lw=1,label=f'Mean ({rates.mean():.1f}%)')
ax_c.set_ylabel('30-day readmission rate (%)'); ax_c.set_title('C  Readmission by payer', fontweight='bold')
ax_c.tick_params(axis='x',rotation=20); ax_c.legend(fontsize=8)
for bar,val in zip(bars,rates.values):
    ax_c.text(bar.get_x()+bar.get_width()/2, val+0.3, f'{val:.1f}%',
              ha='center',va='bottom',fontsize=8)

# Panel D — Age violin by admit type
ax_d = fig.add_subplot(gs[1, 0:2])
order_admit=['Emergency','Urgent','Elective','Newborn']
sns.violinplot(data=df,x='admit_type',y='age',order=order_admit,
               palette='Set3',inner='quartile',ax=ax_d)
ax_d.axhline(65,color='red',ls='--',lw=1.2,label='Medicare threshold (65)')
ax_d.set_title('D  Age distribution by admission type', fontweight='bold')
ax_d.set_xlabel(''); ax_d.legend(fontsize=9)

# Panel E — Comorbidity prevalence
ax_e = fig.add_subplot(gs[1, 2])
comorbs = ['diabetes','hypertension','ckd','heart_failure']
rates_c  = df[comorbs].mean()*100
colors_c = ['#6ACC65','#4878CF','#D65F5F','#B47CC7']
bars_c   = ax_e.barh(comorbs, rates_c.values, color=colors_c, edgecolor='white')
for bar,val in zip(bars_c,rates_c.values):
    ax_e.text(val+0.5, bar.get_y()+bar.get_height()/2,
              f'{val:.1f}%',va='center',fontsize=9)
ax_e.set_xlabel('Prevalence (%)'); ax_e.set_title('E  Comorbidity prevalence', fontweight='bold')
ax_e.set_xlim(0,65)

fig.suptitle('Hospital Discharge Cohort — EDA Summary (N=800)', fontsize=14, y=1.01)
plt.savefig('/tmp/mod02/multi_panel_eda.png', bbox_inches='tight', dpi=300)
plt.show()
print("Saved at 300 dpi: multi_panel_eda.png")


## 7. Knowledge Check
1. Why do we annotate clinical reference ranges on lab histograms?
2. When is a violin plot more informative than a box plot?
3. Panel B shows Self-pay has the shortest median LOS. What are two alternative explanations?
4. The creatinine histogram shows strong right skew. Which correlation method should you use when correlating creatinine with LOS?
5. Add a 6th panel to the multi-panel figure showing HbA1c distribution split by `diabetes` flag.


In [ ]:
# Exercise 5 — HbA1c by diabetes flag
fig, ax = plt.subplots(figsize=(7,4))
for flag, label, color in [(0,'No diabetes','#4878CF'),(1,'Diabetes','#D65F5F')]:
    subset = df.loc[df['diabetes']==flag,'hba1c'].dropna()
    sns.kdeplot(subset, ax=ax, label=f'{label} (n={len(subset)})', color=color, fill=True, alpha=0.3)
ax.axvline(6.5,color='black',ls='--',lw=1.2,label='Diagnostic threshold (6.5%)')
ax.set_xlabel('HbA1c (%)'); ax.set_title('HbA1c distribution by diabetes status')
ax.legend(); plt.tight_layout(); plt.show()


---
**Next → NB-04: Time-Series of Admissions and Events**